In [ ]:
"""
NowcastPNN Models for Birth Count Prediction
Adapted for Mexican birth registration data with multiple categorical variables
"""

import torch
import torch.nn as nn


class BirthNowcastPNN(nn.Module):
    """
    Probabilistic Neural Network for nowcasting birth counts.
    
    This model handles multiple categorical variables:
    - Municipality (group_id)
    - Sex (male/female) 
    - Age groups (5-year intervals)
    
    Architecture:
    1. Embedding layers for categorical variables
    2. Attention mechanism for temporal patterns
    3. Convolutional layers for delay patterns
    4. Fully connected layers for final prediction
    5. Negative Binomial output distribution
    """
    
    def __init__(self, 
                 past_units=36,           # Number of past months to consider
                 max_delay=96,            # Maximum delay in months (8 years)
                 n_municipalities=520,    # Number of unique municipalities in Mexico
                 n_age_groups=12,         # Number of age groups (pre-calculated in data)
                 hidden_units=[32, 16],   # Hidden layer dimensions
                 conv_channels=[32, 16, 1], # Convolutional channels
                 embedding_dims={         # Embedding dimensions for categorical variables
                     'municipality': 25,  # ~520/20 for municipalities
                     'sex': 2,
                     'age': 8
                 },
                 dropout_probs=[0.2, 0.15],  # Dropout probabilities
                 load_embeddings=False):     # Whether to load pre-trained embeddings
        
        super().__init__()
        
        # Store dimensions
        self.past_units = past_units
        self.max_delay = max_delay
        self.final_dim = past_units
        
        # Embedding layers for categorical variables
        self.embed_municipality = nn.Embedding(n_municipalities, embedding_dims['municipality'])
        self.embed_sex = nn.Embedding(2, embedding_dims['sex'])  # 0: female, 1: male
        self.embed_age = nn.Embedding(n_age_groups, embedding_dims['age'])
        
        # Calculate total embedding dimension
        total_embed_dim = sum(embedding_dims.values())
        
        # Embedding processing layers
        self.fc_embed1 = nn.Linear(total_embed_dim, 2 * total_embed_dim)
        self.fc_embed2 = nn.Linear(2 * total_embed_dim, past_units)
        self.bnorm_embed = nn.BatchNorm1d(2 * total_embed_dim)
        
        # Attention mechanism for temporal patterns
        self.attn1 = nn.MultiheadAttention(embed_dim=self.max_delay, num_heads=4, batch_first=True)
        self.fc_attn = nn.Linear(self.past_units, self.past_units)
        
        # Convolutional layers for delay pattern extraction
        self.conv1 = nn.Conv1d(self.max_delay, conv_channels[0], kernel_size=7, padding="same")
        self.conv2 = nn.Conv1d(conv_channels[0], conv_channels[1], kernel_size=7, padding="same")
        self.conv3 = nn.Conv1d(conv_channels[1], conv_channels[2], kernel_size=7, padding="same")
        
        # Batch normalization for convolutional layers
        self.bnorm_conv1 = nn.BatchNorm1d(self.max_delay)
        self.bnorm_conv2 = nn.BatchNorm1d(conv_channels[0])
        self.bnorm_conv3 = nn.BatchNorm1d(conv_channels[1])
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.final_dim, hidden_units[0])
        self.fc2 = nn.Linear(hidden_units[0], hidden_units[1])
        
        # CORRECTED NEGATIVE BINOMIAL: Mean head + municipality-specific alpha parameters
        # Formula: Var = μ + α_l·μ² (quadratic variance growth)
        self.mean_head = nn.Linear(hidden_units[-1], 1)     # log μ - mean in log-space
        self.alpha_embedding = nn.Embedding(n_municipalities, 1)  # α_l per municipality (overdispersion)
        
        # Batch normalization for FC layers
        self.bnorm_fc1 = nn.BatchNorm1d(self.final_dim)
        self.bnorm_fc2 = nn.BatchNorm1d(hidden_units[0])
        self.bnorm_final = nn.BatchNorm1d(hidden_units[-1])
        
        # Dropout layers
        self.dropout1 = nn.Dropout(dropout_probs[0])
        self.dropout2 = nn.Dropout(dropout_probs[1])
        
        # Activation functions
        self.act = nn.SiLU()  # Swish activation
        self.softplus = nn.Softplus()
        
        # Scaling constant for birth counts
        self.const = 1000.0  # Adjust based on typical birth count scale
        
        # Initialize embeddings if specified
        if load_embeddings:
            self._load_pretrained_embeddings()
        
        # Initialize alpha embeddings to small positive values for near-Poisson start
        # log(α) ≈ -3 → α ≈ 0.05 (small overdispersion, near Poisson initially)
        nn.init.constant_(self.alpha_embedding.weight, -3.0)
        
        # Initialize mean head normally
        nn.init.normal_(self.mean_head.weight, 0, 0.01)
    
    def _load_pretrained_embeddings(self):
        """Load pre-trained embeddings if available."""
        try:
            # Try to load municipality embeddings
            muni_weights = torch.load("./weights/municipality_embeddings.pt", map_location='cpu')
            self.embed_municipality.weight.data = muni_weights
            print("✓ Loaded pre-trained municipality embeddings")
        except FileNotFoundError:
            print("! Municipality embeddings not found, using random initialization")
    
    def save_embeddings(self):
        """Save trained embeddings for future use."""
        import os
        os.makedirs("./weights", exist_ok=True)
        torch.save(self.embed_municipality.weight.data, "./weights/municipality_embeddings.pt")
        print("✓ Embeddings saved to ./weights/")
    
    def forward(self, reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset=None):
        """
        Forward pass with population offset support.
        
        Args:
            reporting_triangle: Tensor of shape [batch, past_units, max_delay]
            municipality_ids: Tensor of shape [batch] with municipality indices
            sex_ids: Tensor of shape [batch] with sex indices (0=female, 1=male)
            age_group_ids: Tensor of shape [batch] with age group indices
            population_offset: Tensor of shape [batch] with log population offsets (optional)
            
        Returns:
            Independent Negative Binomial distribution
        """
        
        # Process reporting triangle
        x = reporting_triangle.float()
        batch_size = x.size(0)
        
        # === EMBEDDING PROCESSING ===
        # Handle scalar inputs by converting to batch format
        if municipality_ids.dim() == 0:
            municipality_ids = municipality_ids.unsqueeze(0)
        if sex_ids.dim() == 0:
            sex_ids = sex_ids.unsqueeze(0)
        if age_group_ids.dim() == 0:
            age_group_ids = age_group_ids.unsqueeze(0)
        
        # Get embeddings
        embed_muni = self.embed_municipality(municipality_ids)
        embed_sex = self.embed_sex(sex_ids)
        embed_age = self.embed_age(age_group_ids)
        
        # Concatenate all embeddings
        embeddings = torch.cat([embed_muni, embed_sex, embed_age], dim=1)
        
        # Process embeddings through FC layers
        embed_processed = self.act(self.fc_embed1(embeddings))
        embed_processed = self.bnorm_embed(embed_processed)
        embed_processed = self.act(self.fc_embed2(embed_processed))
        
        # === ATTENTION MECHANISM ===
        x_residual = x.clone()
        
        # Self-attention on temporal dimension
        x_attn, _ = self.attn1(x, x, x, need_weights=False)
        
        # Process through FC layer and add residual connection
        x_attn = self.act(self.fc_attn(x_attn.permute(0, 2, 1)))
        x = x_attn.permute(0, 2, 1) + x_residual
        
        # === CONVOLUTIONAL PROCESSING ===
        # Reshape for convolution: [batch, max_delay, past_units]
        x = x.permute(0, 2, 1)
        
        # Apply convolutional layers with batch norm and activation
        x = self.act(self.conv1(self.bnorm_conv1(x)))
        x = self.act(self.conv2(self.bnorm_conv2(x)))
        x = self.act(self.conv3(self.bnorm_conv3(x)))
        
        # Remove the delay dimension (should be 1 after final conv)
        x = torch.squeeze(x, dim=1)
        
        # === ADD CATEGORICAL INFORMATION ===
        # Add processed embeddings to the main features
        x = x + embed_processed
        
        # === FULLY CONNECTED LAYERS ===
        x = self.dropout1(x)
        x = self.act(self.fc1(self.bnorm_fc1(x)))
        
        x = self.dropout2(x)
        features = self.act(self.fc2(self.bnorm_fc2(x)))  # Final feature representation
        
        # === CORRECTED NEGATIVE BINOMIAL OUTPUT ===
        # FORMULA: Var = μ + α_l·μ² (quadratic variance growth)
        
        # Mean prediction in log-space
        log_birth_rate = self.mean_head(self.bnorm_final(features)).squeeze(-1)     # log μ
        
        # Add population offset to mean if provided (GLM-style)
        if population_offset is not None:
            log_births_mean = log_birth_rate + population_offset  # log(μ) = log_rate + log(pop)
        else:
            log_births_mean = log_birth_rate
            
        # Convert to count-space mean with floor for numerical stability
        births_mean = torch.clamp(torch.exp(log_births_mean), min=1e-8)  # μ ≥ 1e-8
        
        # Get municipality-specific overdispersion parameters α_l
        log_alpha = self.alpha_embedding(municipality_ids).squeeze(-1)  # log(α_l)
        alpha_l = self.softplus(log_alpha) + 1e-6  # α_l > 0, ensure numerical stability
        
        # CORRECTED Negative Binomial variance: Var = μ + α_l·μ²
        births_variance = births_mean + alpha_l * births_mean * births_mean
        
        # CORRECTED Convert to PyTorch NB parameterization
        # Formula: Var = μ + α_l·μ²
        # NB parameterization: r = 1/α_l (independent of μ), p = r/(r + μ)
        r = 1.0 / (alpha_l + 1e-8)  # total_count = r = 1/α_l
        p = births_mean / (r + births_mean)

        
        # Clamp parameters for numerical stability
        total_count = torch.clamp(r, min=1e-6, max=1e6)
        probs = torch.clamp(p, min=1e-6, max=1.0 - 1e-6)
        
        # Create Negative Binomial distribution
        # This models: births ~ NB with Var = μ + α_l·μ²
        dist = torch.distributions.NegativeBinomial(total_count=total_count, probs=probs)
        # --- DEBUG TEST ---
        if torch.isnan(dist.mean).any():
            print("NaN in mean!")
        print(
            "Example μ:", births_mean[0].item(),
            "α:", alpha_l[0].item(),
            "Var:", dist.variance[0].item()
        )
        # Return as independent distribution
        return torch.distributions.Independent(dist, reinterpreted_batch_ndims=1)


class SimpleBirthNowcastPNN(nn.Module):
    """
    Simplified version of BirthNowcastPNN for faster experimentation.
    
    This version has fewer parameters and simpler architecture while maintaining
    the core functionality for birth count nowcasting.
    """
    
    def __init__(self, 
                 past_units=36,
                 max_delay=96,
                 n_municipalities=520,    # Updated for Mexico's 520 municipalities
                 n_age_groups=12,
                 embedding_dims={'municipality': 64, 'sex': 2, 'age': 8},  # Increased municipality: 20->64, age: 6->8
                 hidden_units=[16, 8]):
        
        super().__init__()
        
        self.past_units = past_units
        self.max_delay = max_delay
        
        # Simplified embeddings
        self.embed_municipality = nn.Embedding(n_municipalities, embedding_dims['municipality'])
        self.embed_sex = nn.Embedding(2, embedding_dims['sex'])
        self.embed_age = nn.Embedding(n_age_groups, embedding_dims['age'])
        
        total_embed_dim = sum(embedding_dims.values())
        
        # Single embedding processing layer
        self.fc_embed = nn.Linear(total_embed_dim, past_units)
        
        # Simplified convolutional processing
        self.conv1 = nn.Conv1d(max_delay, 16, kernel_size=5, padding="same")
        self.conv2 = nn.Conv1d(16, 1, kernel_size=5, padding="same")
        
        # Simplified FC layers
        self.fc1 = nn.Linear(past_units, hidden_units[0])
        self.fc2 = nn.Linear(hidden_units[0], hidden_units[1])
        
        # CORRECTED NEGATIVE BINOMIAL: Mean head + municipality-specific alpha parameters
        self.mean_head = nn.Linear(hidden_units[1], 1)     # log μ - mean in log-space
        self.alpha_embedding = nn.Embedding(n_municipalities, 1)  # α_l per municipality (overdispersion)
        
        # Activations and regularization
        self.act = nn.ReLU()
        self.softplus = nn.Softplus()
        self.dropout = nn.Dropout(0.1)
        
        self.const = 100.0
        
        # Initialize alpha embeddings to small positive values for near-Poisson start
        nn.init.constant_(self.alpha_embedding.weight, -3.0)  # log(α) ≈ -3 → α ≈ 0.05
        nn.init.normal_(self.mean_head.weight, 0, 0.01)
    
    def forward(self, reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset=None):
        """Simplified forward pass with population offset support."""
        
        x = reporting_triangle.float()
        
        # Process embeddings
        embed_muni = self.embed_municipality(municipality_ids)
        embed_sex = self.embed_sex(sex_ids)
        embed_age = self.embed_age(age_group_ids)
        
        embeddings = torch.cat([embed_muni, embed_sex, embed_age], dim=1)
        embed_processed = self.act(self.fc_embed(embeddings))
        
        # Convolutional processing
        x = x.permute(0, 2, 1)  # [batch, max_delay, past_units]
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = torch.squeeze(x, dim=1)
        
        # Add embeddings
        x = x + embed_processed
        
        # FC layers
        x = self.dropout(x)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        features = self.act(self.fc2(x))  # Final feature representation
        
        # === CORRECTED NEGATIVE BINOMIAL OUTPUT ===
        # FORMULA: Var = μ + α_l·μ² (quadratic variance growth)
        
        # Mean prediction in log-space
        log_birth_rate = self.mean_head(features).squeeze(-1)     # log μ
        
        # Add population offset to mean if provided (GLM-style)
        if population_offset is not None:
            log_births_mean = log_birth_rate + population_offset  # log(μ) = log_rate + log(pop)
        else:
            log_births_mean = log_birth_rate
            
        # Convert to count-space mean with floor for numerical stability
        births_mean = torch.clamp(torch.exp(log_births_mean), min=1e-8)  # μ ≥ 1e-8
        
        # Get municipality-specific overdispersion parameters α_l
        log_alpha = self.alpha_embedding(municipality_ids).squeeze(-1)  # log(α_l)
        alpha_l = self.softplus(log_alpha) + 1e-6  # α_l > 0
        
        # CORRECTED Negative Binomial variance: Var = μ + α_l·μ²
        births_variance = births_mean + alpha_l * births_mean * births_mean
        
        # CORRECTED Convert to PyTorch NB parameterization
        r = 1.0 / (alpha_l + 1e-8)
        p = births_mean / (r + births_mean)

        # Clamp for numerical stability
        total_count = torch.clamp(r, min=1e-6, max=1e6)
        probs = torch.clamp(p, min=1e-6, max=1.0 - 1e-6)
        
        # Create Negative Binomial distribution
        dist = torch.distributions.NegativeBinomial(total_count=total_count, probs=probs)
        
        return torch.distributions.Independent(dist, reinterpreted_batch_ndims=1)


#### DATA PROCESSING

-------------

In [ ]:
"""
Data Processing Module for Birth Nowcasting
Handles creation of reporting triangles and data preparation for the NowcastPNN model
"""

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from typing import Dict, Tuple, Optional, List
from datetime import datetime
import warnings
import pyreadr
warnings.filterwarnings('ignore')


class BirthDataProcessor:
    """
    Processes raw birth registration data into reporting triangles for nowcasting.
    
    The reporting triangle is a matrix of size [past_units, max_delay] where:
    - past_units: number of past months to consider
    - max_delay: maximum reporting delay in months
    """
    
    def __init__(self, 
                 past_units: int = 36,
                 max_delay: int = 96,
                 min_births_threshold: int = 10,
                 nowcast_cutoff_year: int = 2016,
                 current_date = None):
        """
        Initialize the data processor.
        
        Args:
            past_units: Number of past months to include in reporting triangles
            max_delay: Maximum reporting delay to consider (months)
            min_births_threshold: Minimum number of births to include a group
            nowcast_cutoff_year: Year from which to start nowcasting (vs training on complete data)
            current_date: Current date for nowcasting calculations (None = use latest data date)
        """
        self.past_units = past_units
        self.max_delay = max_delay
        self.max_delay_months = max_delay
        self.min_births_threshold = min_births_threshold
        self.nowcast_cutoff_year = nowcast_cutoff_year
        self.current_date = current_date
        
        # Label encoders for categorical variables
        self.le_mun = LabelEncoder()
        self.le_sex = LabelEncoder()
        self.le_age = LabelEncoder()
        
        # Normalization baselines (calculated from training period)
        self.municipality_baselines = None
        
        # Population data for offset
        self.population_data = None
        self.use_population_offset = False
        self.use_log_transform = False
        
        # Metadata
        self.n_municipalities = 0
        self.n_sexes = 0
        self.n_age_groups = 0
        
        # Data containers
        self.data = None
        self.municipalities = None
        self.sexes = None
        self.age_groups = None
        self.date_range = None
    
    def filter_reproductive_ages(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Filter to reproductive ages: females 15-44, males 15-69
        """
        print("Filtering to reproductive ages...")
        initial_count = len(df)
        
        # Define reproductive age ranges
        female_ages = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44']
        male_ages = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69']
        
        # Filter by sex and age
        female_mask = (df['sex'] == 'female') & (df['age'].isin(female_ages))
        male_mask = (df['sex'] == 'male') & (df['age'].isin(male_ages))
        
        df_filtered = df[female_mask | male_mask].copy()
        
        print(f"Filtered from {initial_count:,} to {len(df_filtered):,} records")
        return df_filtered
    
    def apply_birth_threshold(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applica una soglia minima sul numero di record per (group_id, sex, age).
        Versione veloce senza set_index/isin.
        """
        print(f"Applying minimum births threshold of {self.min_births_threshold} (by row count)...")
        initial_count = len(df)

        # Sanity check colonne
        for col in ['group_id', 'sex', 'age']:
            if col not in df.columns:
                raise KeyError(f"Column '{col}' not found in dataframe at apply_birth_threshold")

        # Conta righe per gruppo in modo vectorized
        counts = (
            df.groupby(['group_id', 'sex', 'age'], observed=True)
            .size()
            .reset_index(name='n')
        )

        valid = counts.loc[counts['n'] >= self.min_births_threshold, ['group_id', 'sex', 'age']]
        df_filtered = df.merge(valid, on=['group_id', 'sex', 'age'], how='inner')

        print(f"Filtered from {initial_count:,} to {len(df_filtered):,} records")
        print(f"Kept {len(valid):,} groups with ≥ {self.min_births_threshold} rows")
        return df_filtered

    
    def fit_label_encoders(self, df: pd.DataFrame):
        """
        Fit label encoders for categorical variables.
        """
        print("Fitting label encoders...")
        
        # Fit encoders
        self.le_mun.fit(df['group_id'].unique())
        self.le_sex.fit(df['sex'].unique())
        self.le_age.fit(df['age'].unique())
        
        # Store metadata
        self.n_municipalities = len(self.le_mun.classes_)
        self.n_sexes = len(self.le_sex.classes_)
        self.n_age_groups = len(self.le_age.classes_)
        
        # Store unique values
        self.municipalities = list(self.le_mun.classes_)
        self.sexes = list(self.le_sex.classes_)
        self.age_groups = list(self.le_age.classes_)
        
        print(f"Encoded {self.n_municipalities} municipalities")
        print(f"Encoded {self.n_sexes} sex categories: {list(self.le_sex.classes_)}")
        print(f"Encoded {self.n_age_groups} age groups: {list(self.le_age.classes_)}")

    def calculate_municipality_baselines(self, df: pd.DataFrame):
        """
        Calculate baseline birth rates for each municipality-sex-age combination
        using complete historical data (1990-2015) for normalization.
        """
        print("Calculating municipality baselines for normalization...")
        
        # Filter to training period (complete data)
        training_data = df[df['year_occ'] <= 2015].copy()
        
        # Calculate average births per month for each municipality-sex-age combination
        baselines = (training_data.groupby(['group_id', 'sex', 'age'])['births']
                    .mean()
                    .reset_index()
                    .rename(columns={'births': 'baseline'}))
        
        # Add small constant to avoid division by zero
        baselines['baseline'] = baselines['baseline'] + 1.0
        
        # Store as dictionary for fast lookup
        self.municipality_baselines = {}
        for _, row in baselines.iterrows():
            key = (row['group_id'], row['sex'], row['age'])
            self.municipality_baselines[key] = row['baseline']
        
        print(f"Calculated baselines for {len(self.municipality_baselines)} municipality-sex-age combinations")
        print(f"Baseline range: {baselines['baseline'].min():.1f} - {baselines['baseline'].max():.1f}")

    def normalize_births(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Normalize births by municipality baseline to reduce scale differences.
        Optimized version using vectorized operations.
        """
        if self.municipality_baselines is None:
            print("Warning: No baselines calculated. Skipping normalization.")
            return df
        
        print("Applying normalization (vectorized)...")
        
        # Convert baselines dictionary to DataFrame for fast merging
        baseline_df = pd.DataFrame([
            {'group_id': k[0], 'sex': k[1], 'age': k[2], 'baseline': v}
            for k, v in self.municipality_baselines.items()
        ])
        
        # Use merge for fast vectorized lookup (much faster than apply)
        df_norm = df.merge(baseline_df, on=['group_id', 'sex', 'age'], how='left')
        
        # Fill missing baselines with 1.0 and normalize
        df_norm['baseline'] = df_norm['baseline'].fillna(1.0)
        df_norm['births_original'] = df_norm['births']  # Keep original
        df_norm['births'] = df_norm['births'] / df_norm['baseline']  # Vectorized division
        
        # Drop the baseline column (no longer needed)
        df_norm = df_norm.drop(columns=['baseline'])
        
        print(f"Normalization completed. Range: {df_norm['births'].min():.3f} - {df_norm['births'].max():.3f}")
        
        return df_norm

    def load_population_data(self, population_path: str = '../../datasets/population_new_mun.RDS'):
        """
        Load and process population data for offset calculation.
        """
        print("Loading population data...")
        
        try:
            # Load R dataset
            result = pyreadr.read_r(population_path)
            pop_df = result[None]  # Default key for single dataframe
            
            print(f"Raw population data: {len(pop_df):,} records")
            print("Columns:", list(pop_df.columns))
            
            # Clean and standardize column names (adapt based on actual structure)
            if 'year' in pop_df.columns:
                # Convert years to int instead of float
                pop_df['year'] = pop_df['year'].astype(int)
                
            # Identify the correct column names (may need adjustment based on actual data)
            municipality_col = None
            year_col = None
            sex_col = None  
            age_col = None
            population_col = None
            
            for col in pop_df.columns:
                if 'mun' in col.lower() or 'group_id' in col.lower():
                    municipality_col = col
                elif 'year' in col.lower() or 'ano' in col.lower():
                    year_col = col
                elif 'sex' in col.lower() or 'sexo' in col.lower():
                    sex_col = col
                elif 'age' in col.lower() or 'edad' in col.lower():
                    age_col = col
                elif 'pop' in col.lower() or 'poblacion' in col.lower() or col.lower() in ['population', 'n']:
                    population_col = col
            
            print(f"Detected columns: municipality={municipality_col}, year={year_col}, sex={sex_col}, age={age_col}, population={population_col}")
            
            # Rename columns to standard names
            column_mapping = {}
            if municipality_col: column_mapping[municipality_col] = 'group_id'
            if year_col: column_mapping[year_col] = 'year'
            if sex_col: column_mapping[sex_col] = 'sex'
            if age_col: column_mapping[age_col] = 'age'
            if population_col: column_mapping[population_col] = 'population'
            
            pop_df = pop_df.rename(columns=column_mapping)
            
            # Standardize sex values to match birth data
            if 'sex' in pop_df.columns:
                pop_df['sex'] = pop_df['sex'].str.lower().replace({
                    'm': 'male', 'f': 'female', 'h': 'male', 'mujer': 'female', 'hombre': 'male'
                })
            
            # Filter to reproductive ages (same as birth data)
            if 'age' in pop_df.columns and 'sex' in pop_df.columns:
                female_ages = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44']
                male_ages = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69']
                
                # Filter to reproductive ages only
                female_mask = (pop_df['sex'] == 'female') & (pop_df['age'].isin(female_ages))
                male_mask = (pop_df['sex'] == 'male') & (pop_df['age'].isin(male_ages))
                pop_df = pop_df[female_mask | male_mask].copy()
                
                print(f"Filtered to reproductive ages: {len(pop_df):,} records")
            
            # Convert years to int and ensure population is numeric
            if 'year' in pop_df.columns:
                pop_df['year'] = pop_df['year'].fillna(0).astype(int)
            if 'population' in pop_df.columns:
                pop_df['population'] = pd.to_numeric(pop_df['population'], errors='coerce').fillna(0)
            
            # Store processed population data
            self.population_data = pop_df
            self.use_population_offset = True
            
            print(f"✅ Population data loaded: {len(pop_df):,} records")
            print(f"Years available: {sorted(pop_df['year'].unique())}")
            print(f"Municipalities: {pop_df['group_id'].nunique()}")
            
            return pop_df
            
        except Exception as e:
            print(f"❌ Error loading population data: {e}")
            print("Continuing without population offset...")
            self.use_population_offset = False
            return None

    def get_population_offset(self, group_id: str, sex: str, age: str, year: int) -> float:
        """
        Get population offset for a specific municipality-sex-age-year combination.
        """
        if not self.use_population_offset or self.population_data is None:
            return 1.0  # No offset
        
        # Find matching population record
        mask = (
            (self.population_data['group_id'] == group_id) &
            (self.population_data['sex'] == sex) &
            (self.population_data['age'] == age) &
            (self.population_data['year'] == year)
        )
        
        matching_records = self.population_data[mask]
        
        if len(matching_records) > 0:
            population = matching_records['population'].iloc[0]
            return max(population, 1.0)  # Minimum 1 to avoid log(0)
        else:
            # No exact match - try to find closest year or use default
            year_mask = (
                (self.population_data['group_id'] == group_id) &
                (self.population_data['sex'] == sex) &
                (self.population_data['age'] == age)
            )
            year_records = self.population_data[year_mask]
            
            if len(year_records) > 0:
                # Use closest available year
                closest_year_idx = (year_records['year'] - year).abs().idxmin()
                population = year_records.loc[closest_year_idx, 'population']
                return max(population, 1.0)
            else:
                return 100.0  # Default population estimate

    def enable_log_transform(self, enable: bool = True):
        """
        Enable or disable log transformation of birth counts.
        
        NOTE: For Negative Binomial models, set enable=False to work with raw counts.
        For Heteroscedastic Normal models, set enable=True to work in log-space.
        """
        self.use_log_transform = enable
        if enable:
            print(f"✓ Log transformation enabled (for Heteroscedastic Normal)")
        else:
            print(f"✓ Log transformation disabled (for Negative Binomial - raw counts)")

    def transform_births(self, births: np.ndarray) -> np.ndarray:
        """
        Apply log transformation to births if enabled.
        For Negative Binomial: returns raw counts (no transformation)
        For Heteroscedastic Normal: returns log(1 + births)
        """
        if self.use_log_transform:
            return np.log1p(births)  # log(1 + births) for Heteroscedastic Normal
        return births.astype(np.float32)  # Raw counts for Negative Binomial

    def inverse_transform_births(self, transformed_births: np.ndarray) -> np.ndarray:
        """
        Inverse transformation to get back birth counts.
        For Negative Binomial: no transformation needed
        For Heteroscedastic Normal: exp(log_births) - 1
        """
        if self.use_log_transform:
            return np.expm1(transformed_births)  # exp(log_births) - 1
        return transformed_births  # Already raw counts

    def create_reporting_triangles(self, period_filter: str = 'all') -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Create reporting triangles for different periods with log transform and population offset.
        
        Args:
            period_filter: Filter for data period ('training', 'nowcasting', 'all')
            
        Returns:
            Tuple of (triangles, group_ids, sex_codes, age_codes, population_offsets)
        """
        all_triangles = []
        all_group_ids = []
        all_sex_codes = []
        all_age_codes = []
        all_population_offsets = []
        
        print(f"Creating reporting triangles for period: {period_filter}")
        
        for mun_id in self.municipalities:
            mun_data = self.data[self.data['group_id'] == mun_id]
            
            for sex in self.sexes:
                sex_data = mun_data[mun_data['sex'] == sex]
                
                for age_group in self.age_groups:
                    age_data = sex_data[sex_data['age'] == age_group]
                    
                    if len(age_data) == 0:
                        continue
                    
                    # Filter by period if needed
                    if period_filter == 'training':
                        age_data = age_data[age_data['year_occ'] < self.nowcast_cutoff_year]
                    elif period_filter == 'nowcasting':
                        age_data = age_data[age_data['year_occ'] >= self.nowcast_cutoff_year]
                    
                    if len(age_data) == 0:
                        continue
                    
                    # Get population offset for this group
                    # Use middle year of the data for population reference
                    ref_year = int(age_data['year_occ'].median()) if len(age_data) > 0 else 2010
                    population_offset = self.get_population_offset(mun_id, sex, age_group, ref_year)
                    log_population_offset = np.log(population_offset) if self.use_population_offset else 0.0
                    
                    # Create triangle for this group
                    triangle = self._create_triangle_for_group(age_data, period_filter)
                    
                    if triangle is not None:
                        all_triangles.append(triangle)
                        all_group_ids.append(self.le_mun.transform([mun_id])[0])
                        all_sex_codes.append(self.le_sex.transform([sex])[0])
                        all_age_codes.append(self.le_age.transform([age_group])[0])
                        all_population_offsets.append(log_population_offset)
        
        print(f"Created {len(all_triangles)} triangles for {period_filter} period")
        
        return (
            np.array(all_triangles),
            np.array(all_group_ids, dtype=int),
            np.array(all_sex_codes, dtype=int),
            np.array(all_age_codes, dtype=int),
            np.array(all_population_offsets, dtype=np.float32)
        )
    
    def _create_triangle_for_group(self, group_data: pd.DataFrame, period_filter: str = 'all') -> Optional[np.ndarray]:
        """Create reporting triangle for a specific group."""
        if len(group_data) == 0:
            return None
        
        # Calculate month_id and delay if not present
        if 'month_id' not in group_data.columns:
            group_data = group_data.copy()
            group_data['month_id'] = group_data['year_occ'] * 12 + group_data['month_occ']
        
        # Use existing delay if available, otherwise calculate
        if 'delay' not in group_data.columns or group_data['delay'].isna().any():
            group_data = group_data.copy()
            group_data['reg_month_id'] = group_data['year_reg'] * 12 + group_data['month_reg']
            group_data['delay'] = group_data['reg_month_id'] - group_data['month_id']
        
        # Extract months and delays
        group_months = group_data['month_id'].values
        group_delays = group_data['delay'].values
        group_births = group_data['births'].values
        
        # Define triangle dimensions
        M = self.past_units  # Number of months to include
        D = self.max_delay_months  # Maximum delay
        
        # For training period: use all available months
        # For nowcasting period: limit to observable months based on current date
        if period_filter == 'nowcasting' and self.current_date is not None:
            current_month_id = self.current_date.year * 12 + self.current_date.month
            max_observable_month = current_month_id - 1  # Last complete month
        else:
            max_observable_month = group_months.max()
        
        # Select the last M months that are available
        end_month = min(max_observable_month, group_months.max())
        start_month = end_month - M + 1
        
        # Initialize triangle
        triangle = np.zeros((M, D))
        
        # Fill triangle with birth counts (apply log transform if enabled)
        for month, delay, births in zip(group_months, group_delays, group_births):
            if start_month <= month <= end_month and 0 <= delay < D:
                month_idx = int(month - start_month)
                delay_idx = int(delay)
                
                # Apply log transformation if enabled
                births_transformed = self.transform_births(np.array([births]))[0]
                triangle[month_idx, delay_idx] += births_transformed
        
        # For nowcasting period, mask future observations that wouldn't be observable yet
        if period_filter == 'nowcasting' and self.current_date is not None:
            for month_idx in range(M):
                target_month = start_month + month_idx
                max_observable_delay = self._calculate_max_observable_delay(target_month, self.current_date)
                # Mask delays that are not yet observable
                max_delay_int = int(max_observable_delay)
                if max_delay_int < D:
                    triangle[month_idx, max_delay_int+1:] = 0
        
        return triangle
    
    def _calculate_max_observable_delay(self, occurrence_month: int, current_date: datetime) -> int:
        """Calculate the maximum observable delay for a given occurrence month."""
        current_month_id = current_date.year * 12 + current_date.month
        max_delay = current_month_id - occurrence_month - 1  # -1 because we need complete months
        return max(0, min(max_delay, self.max_delay_months))
    
    def get_feature_dimensions(self) -> Dict[str, int]:
        """Get dimensions for embedding layers."""
        return {
            'n_municipalities': self.n_municipalities,
            'n_sexes': self.n_sexes,
            'n_age_groups': self.n_age_groups,
            'triangle_shape': (self.past_units, self.max_delay)
        }


class BirthDataset(Dataset):
    """PyTorch dataset for birth nowcasting data with population offset support."""
    
    def __init__(self, triangles: np.ndarray, group_ids: np.ndarray, 
                 sex_codes: np.ndarray, age_codes: np.ndarray, 
                 population_offsets: Optional[np.ndarray] = None,
                 use_log_transform: bool = False):
        """
        Initialize dataset.
        
        Args:
            triangles: Reporting triangles [N, M, D]
            group_ids: Municipality IDs [N]
            sex_codes: Sex codes [N]
            age_codes: Age group codes [N]
            population_offsets: Log population offsets [N] (optional)
            use_log_transform: Whether to apply log transformation to targets
        """
        self.triangles = torch.FloatTensor(triangles)
        self.group_ids = torch.LongTensor(group_ids)
        self.sex_codes = torch.LongTensor(sex_codes)
        self.age_codes = torch.LongTensor(age_codes)
        self.use_log_transform = use_log_transform
        
        # Population offset (log-transformed)
        if population_offsets is not None:
            self.population_offsets = torch.FloatTensor(population_offsets)
        else:
            self.population_offsets = torch.zeros(len(triangles))
        
    def __len__(self) -> int:
        return len(self.triangles)
    
    def transform_births(self, births: np.ndarray) -> np.ndarray:
        """
        Apply transformation to births if enabled.
        For Negative Binomial: returns raw counts (no transformation)
        For Heteroscedastic Normal: returns log(1 + births)
        """
        if self.use_log_transform:
            return np.log1p(births)  # log(1 + births) for Heteroscedastic Normal
        return births.astype(np.float32)  # Raw counts for Negative Binomial
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Create target births by summing all observed births in the triangle
        target_births = self.triangles[idx].sum()
        
        # Apply log transformation to target births if enabled (for heteroscedastic model)
        if self.use_log_transform:
            target_births = self.transform_births(np.array([target_births]))[0]
        
        return {
            'reporting_triangle': self.triangles[idx],
            'municipality_id': self.group_ids[idx],
            'sex_id': self.sex_codes[idx],
            'age_group_id': self.age_codes[idx],
            'target_births': target_births,
            'population_offset': self.population_offsets[idx]
        }


def create_data_loaders(triangles: np.ndarray, group_ids: np.ndarray,
                       sex_codes: np.ndarray, age_codes: np.ndarray,
                       population_offsets: Optional[np.ndarray] = None,
                       batch_size: int = 32, train_split: float = 0.8,
                       use_log_transform: bool = False) -> Tuple[DataLoader, DataLoader]:
    """Create training and validation data loaders with population offset support."""
    
    # Create dataset
    dataset = BirthDataset(triangles, group_ids, sex_codes, age_codes, 
                          population_offsets, use_log_transform)
    
    # Split into train/val
    n_samples = len(dataset)
    n_train = int(n_samples * train_split)
    
    train_indices = torch.randperm(n_samples)[:n_train]
    val_indices = torch.randperm(n_samples)[n_train:]
    
    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset = torch.utils.data.Subset(dataset, val_indices)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader


-----

#### TRAIN NEGATIVE BINOMIAL

---

In [ ]:
"""
Training with CORRECTED Negative Binomial Distribution
Implementation of municipality-specific overdispersion parameters
CORRECTED formula: Var = μ + α_l·μ² where α_l is municipality-specific
"""

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
from typing import Dict, Any, List, Tuple
import warnings

from data_processing import BirthDataProcessor
from birth_nowcast_models import BirthNowcastPNN

class NegativeBinomialTrainer:
    """
    Trainer for CORRECTED Negative Binomial distribution with municipality-specific overdispersion.
    
    The negative binomial distribution is parameterized as:
    - μ = mean (from neural network)
    - α_l = municipality-specific overdispersion parameter
    - CORRECTED Variance = μ + α_l·μ² (quadratic growth)
    """
    
    def __init__(self, model: nn.Module, device: torch.device):
        self.model = model
        self.device = device
        self.optimizer = None
        self.history = {'train_loss': [], 'val_loss': []}
        
    def negative_binomial_loss(self, dist, targets):
        """
        Compute negative log-likelihood for Negative Binomial distribution.
        
        Args:
            dist: torch.distributions.NegativeBinomial distribution
            targets: actual birth counts (raw counts, not log-transformed)
            
        Returns:
            loss: negative log-likelihood
        """
        # Ensure targets are integers for Negative Binomial
        targets = targets.round().long()
        
        # Compute negative log-likelihood
        log_prob = dist.log_prob(targets)
        
        # Handle edge cases
        valid_mask = torch.isfinite(log_prob)
        if not valid_mask.all():
            warnings.warn(f"Found {(~valid_mask).sum()} non-finite log probabilities")
            log_prob = log_prob[valid_mask]
        
        # Return negative log-likelihood (we minimize loss)
        return -log_prob.mean()
    
    def train_epoch(self, train_loader: DataLoader) -> float:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        
        for batch in train_loader:
            # Move data to device
            reporting_triangle = batch['reporting_triangle'].to(self.device)
            municipality_ids = batch['municipality_id'].squeeze().to(self.device)
            sex_ids = batch['sex_id'].squeeze().to(self.device)
            age_group_ids = batch['age_group_id'].squeeze().to(self.device)
            population_offset = batch['population_offset'].squeeze().to(self.device)
            targets = batch['target_births'].squeeze().to(self.device)
            
            # Forward pass
            dist = self.model(reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset)
            
            # Compute loss
            loss = self.negative_binomial_loss(dist, targets)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        return total_loss / num_batches
    
    def validate_epoch(self, val_loader: DataLoader) -> float:
        """Validate for one epoch."""
        self.model.eval()
        total_loss = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for batch in val_loader:
                # Move data to device
                reporting_triangle = batch['reporting_triangle'].to(self.device)
                municipality_ids = batch['municipality_id'].squeeze().to(self.device)
                sex_ids = batch['sex_id'].squeeze().to(self.device)
                age_group_ids = batch['age_group_id'].squeeze().to(self.device)
                population_offset = batch['population_offset'].squeeze().to(self.device)
                targets = batch['target_births'].squeeze().to(self.device)
                
                # Forward pass
                dist = self.model(reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset)
                
                # Compute loss
                loss = self.negative_binomial_loss(dist, targets)
                
                total_loss += loss.item()
                num_batches += 1
        
        return total_loss / num_batches
    
    def train(self, train_loader: DataLoader, val_loader: DataLoader, 
              num_epochs: int, learning_rate: float = 0.001, patience: int = 5) -> Dict[str, Any]:
        """
        Train the model.
        
        Args:
            train_loader: Training data loader
            val_loader: Validation data loader
            num_epochs: Number of epochs
            learning_rate: Learning rate
            patience: Early stopping patience
            
        Returns:
            training_history: Dictionary with training history
        """
        
        # Initialize optimizer
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.5, patience=3)
        
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        print(f"Starting training for {num_epochs} epochs...")
        print(f"Learning rate: {learning_rate}, Patience: {patience}")
        
        for epoch in range(num_epochs):
            # Train
            train_loss = self.train_epoch(train_loader)
            
            # Validate
            val_loss = self.validate_epoch(val_loader)
            
            # Update learning rate scheduler
            scheduler.step(val_loss)
            
            # Record history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            
            # Print progress
            current_lr = self.optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1:3d}/{num_epochs}: "
                  f"Train Loss: {train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, "
                  f"LR: {current_lr:.6f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = self.model.state_dict().copy()
                
                # Save checkpoint
                os.makedirs('./checkpoints', exist_ok=True)
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': best_model_state,
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'history': self.history
                }, './checkpoints/best_model.pt')
                
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch+1} epochs")
                    break
        
        # Load best model
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            print(f"✓ Loaded best model (val_loss: {best_val_loss:.4f})")
        
        return {
            'best_val_loss': best_val_loss,
            'num_epochs_trained': epoch + 1,
            'history': self.history
        }
    
    def evaluate(self, test_loader: DataLoader) -> Dict[str, Any]:
        """
        Evaluate the model on test data.
        
        Returns:
            results: Dictionary with evaluation metrics
        """
        self.model.eval()
        
        all_predictions = []
        all_targets = []
        all_mean_predictions = []
        all_variances = []
        all_theta_params = []
        total_loss = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for batch in test_loader:
                # Move data to device
                reporting_triangle = batch['reporting_triangle'].to(self.device)
                municipality_ids = batch['municipality_id'].squeeze().to(self.device)
                sex_ids = batch['sex_id'].squeeze().to(self.device)
                age_group_ids = batch['age_group_id'].squeeze().to(self.device)
                population_offset = batch['population_offset'].squeeze().to(self.device)
                targets = batch['target_births'].squeeze().to(self.device)
                
                # Forward pass
                dist = self.model(reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset)
                
                # Compute loss
                loss = self.negative_binomial_loss(dist, targets)
                total_loss += loss.item()
                num_batches += 1
                
                # Collect predictions and statistics
                mean_pred = dist.mean.cpu().numpy()
                variance_pred = dist.variance.cpu().numpy()
                targets_np = targets.cpu().numpy()
                
                all_mean_predictions.extend(mean_pred)
                all_variances.extend(variance_pred)
                all_targets.extend(targets_np)
                
                # Sample from distribution for additional metrics
                samples = dist.sample().cpu().numpy()
                all_predictions.extend(samples)
                
                # Get alpha parameters for municipality analysis
                if hasattr(self.model, 'alpha_embedding'):
                    alpha_vals = torch.nn.functional.softplus(self.model.alpha_embedding(municipality_ids))
                    all_theta_params.extend(alpha_vals.cpu().numpy())
        
        # Convert to numpy arrays
        predictions = np.array(all_predictions)
        targets = np.array(all_targets)
        mean_predictions = np.array(all_mean_predictions)
        variances = np.array(all_variances)
        
        # Calculate metrics
        mae = np.mean(np.abs(mean_predictions - targets))
        mse = np.mean((mean_predictions - targets) ** 2)
        rmse = np.sqrt(mse)
        mape = np.mean(np.abs((targets - mean_predictions) / np.maximum(targets, 1e-8))) * 100
        
        # Overdispersion analysis
        mean_overdispersion = np.mean(variances / np.maximum(mean_predictions, 1e-8))
        theta_stats = {}
        if all_theta_params:
            theta_array = np.array(all_theta_params).flatten()
            theta_stats = {
                'mean': np.mean(theta_array),
                'std': np.std(theta_array),
                'min': np.min(theta_array),
                'max': np.max(theta_array),
                'median': np.median(theta_array)
            }
        
        results = {
            'test_loss': total_loss / num_batches,
            'mae': mae,
            'mse': mse,
            'rmse': rmse,
            'mape': mape,
            'mean_overdispersion': mean_overdispersion,
            'theta_statistics': theta_stats,
            'predictions': mean_predictions,
            'actuals': targets,
            'variances': variances,
            'sample_predictions': predictions  # Sampled predictions
        }
        
        return results


def main():
    """Run training with CORRECTED Negative Binomial distribution."""
    
    print("🎯 CORRECTED NEGATIVE BINOMIAL DISTRIBUTION TRAINING")
    print("=" * 80)
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("Implementation: NegativeBinomial(μ, α_l) with Var = μ + α_l·μ²")
    print("where α_l is municipality-specific overdispersion parameter")
    
    # Configuration
    config = {
        'past_units': 36,
        'max_delay': 96,
        'min_births_threshold': 5,
        'batch_size': 32,
        'num_epochs': 25,  # Slightly more epochs for NB
        'learning_rate': 0.0005,  # Lower learning rate for stability
        'patience': 7,  # More patience for NB convergence
        'nowcast_cutoff_year': 2016,
        'current_date': '2023-12-31'
    }
    
    print("\nConfiguration:")
    for key, value in config.items():
        print(f"  {key}: {value}")
    
    # Step 1: Initialize data processor
    print(f"\n{'='*50}")
    print("STEP 1: DATA PROCESSING")
    print(f"{'='*50}")
    
    processor = BirthDataProcessor(
        past_units=config['past_units'],
        max_delay=config['max_delay'],
        min_births_threshold=config['min_births_threshold'],
        nowcast_cutoff_year=config['nowcast_cutoff_year'],
        current_date=pd.to_datetime(config['current_date'])
    )
    
    # Load data
    data_path = os.path.join('../../datasets', 'monthly_births.parquet')
    print(f"Loading data from: {data_path}")
    
    df = pd.read_parquet(data_path)
    print(f"Dataset shape: {df.shape}")
    
    print("Preprocessing data...")
    df_processed = processor.filter_reproductive_ages(df)

    print("Pre-threshold shape:", df_processed.shape)
    print("Nulls:", df_processed[['group_id','sex','age']].isna().sum().to_dict())
    print("Sample:", df_processed[['group_id','sex','age','births']].head().to_dict(orient='list'))

    df_processed = processor.apply_birth_threshold(df_processed)
    
    # Store data and fit encoders
    processor.data = df_processed
    processor.fit_label_encoders(df_processed)
    
    # Load population data
    processor.load_population_data()
    
    # IMPORTANT: Disable log transformation for Negative Binomial (we need raw counts)
    processor.enable_log_transform(False)
    print("✓ Log transformation disabled - using raw counts for Negative Binomial")
    
    # Calculate municipality baselines
    processor.calculate_municipality_baselines(df_processed)
    
    # Create reporting triangles (training period)
    triangles, group_ids, sex_codes, age_codes, pop_offsets = processor.create_reporting_triangles('training')
    print(f"Training samples: {len(triangles):,}")
    
    if len(triangles) == 0:
        print("❌ No training triangles created! Check data filtering.")
        return
    
    # Create data loaders
    from data_processing import create_data_loaders
    train_loader, val_loader = create_data_loaders(
        triangles=triangles,
        group_ids=group_ids, 
        sex_codes=sex_codes,
        age_codes=age_codes,
        population_offsets=pop_offsets,
        batch_size=config['batch_size'],
        use_log_transform=processor.use_log_transform  # Should be False
    )
    
    test_loader = val_loader  # Use validation as test for now
    
    print(f"Training batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")
    
    # Get data info
    sample_batch = next(iter(train_loader))
    reporting_triangle = sample_batch['reporting_triangle']
    targets = sample_batch['target_births']
    
    print(f"\nData shapes:")
    print(f"  Reporting triangle: {reporting_triangle.shape}")
    print(f"  Raw count targets: {targets.shape}")
    print(f"  Target range: [{targets.min().item():.0f}, {targets.max().item():.0f}]")
    print(f"  Target mean: {targets.mean().item():.2f}")
    print(f"  Target data type: {targets.dtype}")
    
    # Step 2: Create Negative Binomial model
    print(f"\n{'='*50}")
    print("STEP 2: MODEL ARCHITECTURE")
    print(f"{'='*50}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = BirthNowcastPNN(
        past_units=config['past_units'],
        max_delay=config['max_delay'],
        n_municipalities=520,
        n_age_groups=12,
        hidden_units=[32, 16],
        embedding_dims={'municipality': 25, 'sex': 2, 'age': 8},
        dropout_probs=[0.2, 0.15]
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Model: BirthNowcastPNN with Negative Binomial Distribution")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Architecture:")
    print(f"  - Mean head: Predicts μ via exp(NN(X) + log(population))")
    print(f"  - Theta embedding: 520 municipality-specific θ_l parameters")
    print(f"  - Distribution: NegativeBinomial(μ, θ_l) with Var = μ(1 + θ_l)")
    
    # Step 3: Test forward pass
    print(f"\n{'='*50}")
    print("STEP 3: FORWARD PASS TEST")
    print(f"{'='*50}")
    
    # Test forward pass
    reporting_triangle = sample_batch['reporting_triangle'].to(device)
    municipality_ids = sample_batch['municipality_id'].squeeze().to(device)
    sex_ids = sample_batch['sex_id'].squeeze().to(device)
    age_group_ids = sample_batch['age_group_id'].squeeze().to(device)
    population_offset = sample_batch['population_offset'].squeeze().to(device)
    targets = sample_batch['target_births'].squeeze().to(device)
    
    print("Testing forward pass...")
    try:
        with torch.no_grad():
            dist = model(reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset)
            
            predicted_mean = dist.mean
            predicted_variance = dist.variance
            
            print(f"✓ Forward pass successful!")
            print(f"  Distribution type: {type(dist)}")
            print(f"  Predicted mean shape: {predicted_mean.shape}")
            print(f"  Predicted mean range: [{predicted_mean.min().item():.2f}, {predicted_mean.max().item():.2f}]")
            print(f"  Predicted variance shape: {predicted_variance.shape}")
            print(f"  Predicted variance range: [{predicted_variance.min().item():.2f}, {predicted_variance.max().item():.2f}]")
            
            # Check overdispersion - CORRECTED formula: Var/μ = 1 + α_l·μ
            overdispersion = predicted_variance / predicted_mean
            print(f"  Overdispersion range: [{overdispersion.min().item():.3f}, {overdispersion.max().item():.3f}]")
            print(f"  Mean overdispersion: {overdispersion.mean().item():.3f}")
            
            # Test alpha parameters (renamed from theta)
            alpha_values = torch.nn.functional.softplus(model.alpha_embedding(municipality_ids))
            print(f"  Alpha parameters range: [{alpha_values.min().item():.6f}, {alpha_values.max().item():.6f}]")
            print(f"  (Formula: Var = μ + α_l·μ²)")
            
    except Exception as e:
        print(f"❌ Forward pass failed: {e}")
        return
    
    # Step 4: Initialize trainer
    print(f"\n{'='*50}")
    print("STEP 4: TRAINER & LOSS FUNCTION")
    print(f"{'='*50}")
    
    trainer = NegativeBinomialTrainer(model=model, device=device)
    
    print(f"✓ NegativeBinomialTrainer initialized")
    print(f"  Loss function: Negative log-likelihood of NegativeBinomial")
    print(f"  Optimizer: Adam with weight decay")
    print(f"  Learning rate scheduler: ReduceLROnPlateau")
    
    # Test loss computation
    try:
        with torch.no_grad():
            dist = model(reporting_triangle, municipality_ids, sex_ids, age_group_ids, population_offset)
            loss = trainer.negative_binomial_loss(dist, targets)
            print(f"  Test loss: {loss.item():.4f}")
            
            if torch.isfinite(loss):
                print(f"  ✓ Loss computation successful")
            else:
                print(f"  ❌ Loss is not finite: {loss.item()}")
                return
                
    except Exception as e:
        print(f"❌ Loss computation failed: {e}")
        return
    
    # Step 5: Training
    print(f"\n{'='*50}")
    print("STEP 5: TRAINING")
    print(f"{'='*50}")
    
    print("Starting training...")
    training_results = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=config['num_epochs'],
        learning_rate=config['learning_rate'],
        patience=config['patience']
    )
    
    print(f"\nTraining completed!")
    print(f"Best validation loss: {training_results['best_val_loss']:.4f}")
    print(f"Epochs trained: {training_results['num_epochs_trained']}")
    
    # Step 6: Evaluation
    print(f"\n{'='*50}")
    print("STEP 6: EVALUATION")
    print(f"{'='*50}")
    
    results = trainer.evaluate(test_loader)
    
    print("Test Results:")
    print(f"  Test Loss: {results['test_loss']:.4f}")
    print(f"  MAE: {results['mae']:.4f}")
    print(f"  RMSE: {results['rmse']:.4f}")
    print(f"  MAPE: {results['mape']:.2f}%")
    print(f"  Mean Overdispersion: {results['mean_overdispersion']:.3f}")
    
    # Theta parameter analysis
    if results['theta_statistics']:
        theta_stats = results['theta_statistics']
        print(f"\nTheta Parameter Analysis (Municipality Overdispersion):")
        print(f"  Mean: {theta_stats['mean']:.6f}")
        print(f"  Std: {theta_stats['std']:.6f}")
        print(f"  Range: [{theta_stats['min']:.6f}, {theta_stats['max']:.6f}]")
        print(f"  Median: {theta_stats['median']:.6f}")
    
    # Save complete model and results for nowcasting
    print(f"\n{'='*50}")
    print("STEP 7: SAVING MODEL & RESULTS")
    print(f"{'='*50}")
    
    # Save model with all metadata
    final_model_path = './checkpoints/negative_binomial_model_final.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_config': {
            'past_units': config['past_units'],
            'max_delay': config['max_delay'],
            'n_municipalities': 520,
            'n_age_groups': 12,
            'hidden_units': [32, 16],
            'embedding_dims': {'municipality': 25, 'sex': 2, 'age': 8},
            'dropout_probs': [0.2, 0.15]
        },
        'training_config': config,
        'training_results': training_results,
        'test_results': results,
        'data_processor_state': {
            'municipality_encoder': processor.le_mun,
            'sex_encoder': processor.le_sex,
            'age_encoder': processor.le_age,
            'use_log_transform': processor.use_log_transform,
            'municipality_baselines': processor.municipality_baselines
        },
        'training_period': f"1990-{config['nowcast_cutoff_year']-1}",
        'model_type': 'negative_binomial',
        'distribution_formula': 'Var = μ(1 + θ_l)',
        'saved_at': datetime.now().isoformat()
    }, final_model_path)
    
    print(f"✅ Final model saved to: {final_model_path}")
    print(f"   📊 Training period: 1990-{config['nowcast_cutoff_year']-1}")
    print(f"   🎯 Model type: Negative Binomial with municipality-specific θ_l")
    print(f"   📈 Formula: Var = μ(1 + θ_l)")
    print(f"   📋 Test MAE: {results['mae']:.2f} births")
    print(f"   📋 Test RMSE: {results['rmse']:.2f} births")
    print(f"   📋 Mean Overdispersion: {results['mean_overdispersion']:.3f}x")
    
    # Save summary for easy access
    summary_path = './checkpoints/training_summary.txt'
    with open(summary_path, 'w') as f:
        f.write(f"NEGATIVE BINOMIAL BIRTH NOWCASTING MODEL\n")
        f.write(f"{'='*50}\n\n")
        f.write(f"Training completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Training period: 1990-{config['nowcast_cutoff_year']-1}\n")
        f.write(f"Model architecture: BirthNowcastPNN with NegativeBinomial distribution\n")
        f.write(f"Municipality-specific parameters: 520 θ_l values\n")
        f.write(f"Variance formula: Var = μ(1 + θ_l)\n\n")
        f.write(f"PERFORMANCE METRICS:\n")
        f.write(f"Test Loss: {results['test_loss']:.4f}\n")
        f.write(f"MAE: {results['mae']:.2f} births\n")
        f.write(f"RMSE: {results['rmse']:.2f} births\n")
        f.write(f"MAPE: {results['mape']:.1f}%\n")
        f.write(f"Mean Overdispersion: {results['mean_overdispersion']:.3f}x\n\n")
        f.write(f"THETA PARAMETERS:\n")
        if results['theta_statistics']:
            theta_stats = results['theta_statistics']
            f.write(f"Mean: {theta_stats['mean']:.6f}\n")
            f.write(f"Std: {theta_stats['std']:.6f}\n")
            f.write(f"Range: [{theta_stats['min']:.6f}, {theta_stats['max']:.6f}]\n")
        f.write(f"\nMODEL FILES:\n")
        f.write(f"Final model: {final_model_path}\n")
        f.write(f"Best checkpoint: ./checkpoints/best_model.pt\n")
    
    print(f"✅ Training summary saved to: {summary_path}")
    print(f"\n🎯 MODEL READY FOR NOWCASTING!")
    print(f"   Use the saved model for validation and nowcast predictions")
    
    print(f"\n✅ Training completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    return results


if __name__ == "__main__":
    main()


--------

#### RUN NOWCAST

------

In [ ]:
"""
Nowcast execution using a trained Negative Binomial model (fast version)
- Loads a saved model and generates predictions for incomplete data (2016–2023)
- Vectorizes sampling (no MC-dropout at inference)
- Caches population offsets to avoid repeated DataFrame scans
"""

import os
import math
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from data_processing import BirthDataProcessor
from birth_nowcast_models import BirthNowcastPNN


def months_between(y1: int, m1: int, y2: int, m2: int) -> int:
    """Inclusive month difference: (y2,m2) - (y1,m1) in months, but not negative."""
    return max(0, (y2 - y1) * 12 + (m2 - m1))


def build_triangle_for_nowcast(df_all: pd.DataFrame,
                               P: int,
                               D: int,
                               y_star: int,
                               m_star: int,
                               group_id: str,
                               sex: str,
                               age: str,
                               max_obs_delay: int) -> np.ndarray:
    """
    Build a P x D reporting triangle for a single (y_star, m_star, gid, sex, age).
    Only delays <= max_obs_delay are filled for the most recent month; the historical
    rows naturally have more complete delays.
    """
    tri = np.zeros((P, D), dtype=np.float32)
    ref_id = y_star * 12 + m_star

    g = df_all[(df_all["group_id"] == group_id) &
               (df_all["sex"] == sex) &
               (df_all["age"] == age)]

    if g.empty:
        return tri

    for i in range(P):
        occ_id = ref_id - i
        y_i = occ_id // 12
        m_i = occ_id % 12
        # pandas modulo of negatives can be odd; normalize month to 1..12
        if m_i == 0:
            y_i -= 1
            m_i = 12

        gi = g[(g["year_occ"] == y_i) & (g["month_occ"] == m_i)]
        if gi.empty:
            continue

        # clamp to [0, D-1] and to current observability for the top row
        if i == 0:
            gi = gi[(gi["delay"] >= 0) & (gi["delay"] <= max_obs_delay) & (gi["delay"] < D)]
        else:
            gi = gi[(gi["delay"] >= 0) & (gi["delay"] < D)]

        if gi.empty:
            continue

        by_d = gi.groupby("delay", as_index=False)["births"].sum()
        tri[i, by_d["delay"].astype(int).to_numpy()] = by_d["births"].astype(np.float32).to_numpy()

    return tri


def observed_so_far(df_all: pd.DataFrame,
                    y_star: int,
                    m_star: int,
                    group_id: str,
                    sex: str,
                    age: str,
                    max_obs_delay: int) -> float:
    """Sum observed registrations so far for the month (delays 0..max_obs_delay)."""
    mask = ((df_all["year_occ"] == y_star) &
            (df_all["month_occ"] == m_star) &
            (df_all["group_id"] == group_id) &
            (df_all["sex"] == sex) &
            (df_all["age"] == age) &
            (df_all["delay"] >= 0) &
            (df_all["delay"] <= max_obs_delay))
    return float(df_all.loc[mask, "births"].sum())


def main():
    print("🎯 BIRTH NOWCAST PREDICTION (fast)")
    print("=" * 60)
    print(f"Started at: {datetime.now():%Y-%m-%d %H:%M:%S}")

    # ---------- Load trained checkpoint ----------
    model_path = os.environ.get("NOWCAST_CHECKPOINT", "./checkpoints/negative_binomial_model_final.pt")
    print(f"Loading model from: {model_path}")

    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    model_config = checkpoint["model_config"]
    training_config = checkpoint["training_config"]
    processor_state = checkpoint["data_processor_state"]

    # ---------- Recreate data processor (same settings as training) ----------
    processor = BirthDataProcessor(
        past_units=processor_state["past_units"],
        max_delay=processor_state["max_delay"],
        min_births_threshold=processor_state["min_births_threshold"],
        nowcast_cutoff_year=processor_state["nowcast_cutoff_year"],
        current_date=pd.to_datetime(training_config["current_date"]),
    )

    # Load original data and apply same filters used in training
    data_path = os.path.join("../../datasets", "monthly_births.parquet")
    df = pd.read_parquet(data_path)

    # We only need 2016–2023 for nowcasting
    df = df[(df["year_occ"] >= 2016) & (df["year_occ"] <= 2023)]
    df = processor.filter_reproductive_ages(df)
    df = processor.apply_birth_threshold(df)

    # Restore encoders and baselines
    processor.le_mun = processor_state["le_municipality"]
    processor.le_sex = processor_state["le_sex"]
    processor.le_age = processor_state["le_age"]
    processor.municipality_baselines = processor_state["municipality_baselines"]
    processor.enable_log_transform(False)

    # Population offset (optional, if available)
    processor.load_population_data()

    # ---------- Build model ----------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BirthNowcastPNN(
        past_units=model_config["past_units"],
        max_delay=model_config["max_delay"],
        n_municipalities=model_config["n_municipalities"],
        n_age_groups=model_config["n_age_groups"],
        hidden_units=model_config["hidden_units"],
        embedding_dims=model_config["embedding_dims"],
        dropout_probs=model_config["dropout_probs"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()  # IMPORTANT: keep dropout OFF at inference

    # ---------- Prepare data frame with delays ----------
    D = processor.max_delay
    P = processor.past_units
    cutoff_dt = processor.current_date

    df_all = df.copy()
    # enforce numeric
    for col in ["year_occ", "month_occ", "year_reg", "month_reg"]:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

    df_all["occ_id"] = df_all["year_occ"] * 12 + df_all["month_occ"]
    df_all["reg_id"] = df_all["year_reg"] * 12 + df_all["month_reg"]
    df_all["delay"] = df_all["reg_id"] - df_all["occ_id"]

    # ---------- Build target grid (months 2016–2023 that are incomplete) ----------
    # We consider each (year, month) and all group/sex/age combos present in that month.
    target_rows: List[Tuple[int, int, str, str, str, int]] = []
    for y in range(2016, 2024):
        for m in range(1, 13):
            # Observable delay for that occurrence month, given the current_date
            max_obs = min(D - 1, months_between(y, m, cutoff_dt.year, cutoff_dt.month))
            if max_obs <= 0:
                continue  # nothing registered yet

            month_g = df_all[(df_all["year_occ"] == y) & (df_all["month_occ"] == m)]
            if month_g.empty:
                continue

            combos = month_g[["group_id", "sex", "age"]].drop_duplicates()
            for _, r in combos.iterrows():
                target_rows.append((y, m, r["group_id"], r["sex"], r["age"], max_obs))

    print(f"✓ Targets to predict: {len(target_rows):,}")

    # ---------- Predict in batches ----------
    mc_samples = max(0, int(training_config.get("nowcast_mc_samples", 200)))
    # batch size: much larger at inference
    batch_size = min(8192, max(1024, int(training_config.get("batch_size", 32)) * 8))

    records: List[Dict] = []
    pop_cache: Dict[Tuple[str, str, str, int], float] = {}

    # Pre-check population usage
    use_pop = getattr(processor, "use_population_offset", False)

    # loop
    for i0 in range(0, len(target_rows), batch_size):
        chunk = target_rows[i0:i0 + batch_size]

        tris = []
        group_ids = []
        sex_ids = []
        age_ids = []
        pop_logs = []
        meta = []

        for (y, m, gid, sex, age, max_obs) in chunk:
            # skip unknown labels (shouldn't happen if we used training filters)
            if ((gid not in processor.le_mun.classes_) or
                (sex not in processor.le_sex.classes_) or
                (age not in processor.le_age.classes_)):
                continue

            tri = build_triangle_for_nowcast(df_all, P, D, y, m, gid, sex, age, max_obs)
            tris.append(tri)
            group_ids.append(int(processor.le_mun.transform([gid])[0]))
            sex_ids.append(int(processor.le_sex.transform([sex])[0]))
            age_ids.append(int(processor.le_age.transform([age])[0]))

            if use_pop:
                key = (gid, sex, age, y)
                if key in pop_cache:
                    pop_val = pop_cache[key]
                else:
                    pop_val = processor.get_population_offset(gid, sex, age, y)
                    pop_cache[key] = pop_val
                pop_logs.append(float(np.log(max(pop_val, 1.0))))
            else:
                pop_logs.append(0.0)

            meta.append((y, m, gid, sex, age, max_obs))

        if not tris:
            continue

        # tensors
        X = torch.tensor(np.stack(tris), dtype=torch.float32, device=device)
        MID = torch.tensor(group_ids, dtype=torch.long, device=device)
        SID = torch.tensor(sex_ids, dtype=torch.long, device=device)
        AID = torch.tensor(age_ids, dtype=torch.long, device=device)
        POP = torch.tensor(pop_logs, dtype=torch.float32, device=device)

        with torch.no_grad():
            # one forward pass; sample observation noise from the NB dist
            dist = model(X, MID, SID, AID, POP)
            if mc_samples > 1:
                # (mc, batch) -> (batch, mc)
                samples = dist.sample((mc_samples,)).permute(1, 0).cpu().numpy()
            else:
                # 0 or 1 => just use the predictive mean, but add a few draws to build an interval
                samples = dist.sample((64,)).permute(1, 0).cpu().numpy()

        mean_pred = samples.mean(axis=1)
        std_pred = samples.std(axis=1)
        lo = np.percentile(samples, 2.5, axis=1)
        hi = np.percentile(samples, 97.5, axis=1)

        # collect
        for j, (y, m, gid, sex, age, max_obs) in enumerate(meta):
            obs = observed_so_far(df_all, y, m, gid, sex, age, max_obs)
            records.append({
                "year_occ": y,
                "month_occ": m,
                "municipality": gid,
                "sex": sex,
                "age_group": age,
                "max_observable_delay": max_obs,
                "observed_so_far": obs,
                "pred_missing_mean": float(mean_pred[j]),
                "pred_missing_std": float(std_pred[j]),
                "pred_missing_lo95": float(lo[j]),
                "pred_missing_hi95": float(hi[j]),
                "total_corrected_mean": float(obs + mean_pred[j]),
                "total_corrected_lo95": float(obs + lo[j]),
                "total_corrected_hi95": float(obs + hi[j]),
            })

    # ---------- Aggregate & save ----------
    pred_df = pd.DataFrame(records)
    if pred_df.empty:
        print("No predictions produced. Check your filters and data paths.")
        return

    # Monthly totals (sum over groups)
    monthly = (pred_df
               .groupby(["year_occ", "month_occ"], as_index=False)
               .agg(registered_births=("observed_so_far", "sum"),
                    estimated_missing=("pred_missing_mean", "sum"),
                    uncertainty_std=("pred_missing_std", "mean")))
    monthly["total_corrected"] = monthly["registered_births"] + monthly["estimated_missing"]
    monthly["percent_estimated"] = 100.0 * monthly["estimated_missing"] / monthly["total_corrected"].clip(lower=1.0)

    # Annual totals
    annual = (monthly
              .groupby("year_occ", as_index=False)
              .agg(registered_births=("registered_births", "sum"),
                   estimated_missing=("estimated_missing", "sum"),
                   uncertainty_std=("uncertainty_std", "mean")))
    annual["total_corrected"] = annual["registered_births"] + annual["estimated_missing"]
    annual["percent_estimated"] = 100.0 * annual["estimated_missing"] / annual["total_corrected"].clip(lower=1.0)
    annual = annual.rename(columns={"year_occ": "year"})

    os.makedirs("./results", exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pred_df.to_csv(f"./results/nowcast_records_{ts}.csv", index=False)
    monthly.to_csv(f"./results/nowcast_monthly_{ts}.csv", index=False)
    annual.to_csv(f"./results/nowcast_annual_{ts}.csv", index=False)

    print("\n📄 Saved:")
    print(f"  ./results/nowcast_records_{ts}.csv")
    print(f"  ./results/nowcast_monthly_{ts}.csv")
    print(f"  ./results/nowcast_annual_{ts}.csv")

    print("\n🧾 Annual summary (2016–2023):")
    for _, row in annual.sort_values('year').iterrows():
        year = int(row["year"])
        reg = row["registered_births"]
        est = row["estimated_missing"]
        tot = row["total_corrected"]
        pct = row["percent_estimated"]
        std = row["uncertainty_std"]
        print(f"  {year}: {reg:,.0f} registered + {est:,.0f} estimated = {tot:,.0f} total "
              f"({pct:.1f}% est) ± {std:,.0f}")

    print(f"\n✅ Nowcast completed at: {datetime.now():%Y-%m-%d %H:%M:%S}")


if __name__ == "__main__":
    main()
